# Dataset 1 - centrality threshold (Gradient Boosting tuned)

Per-dataset version of the centrality p-threshold sweep (replaces the combined `04b`). Same technique as Dataset 2/3's `04_centrality_threshold` - load the p0-selected model and `clone + refit` it on each p's targets - but with Dataset 1's **temporal** split. Writes `predictions/dataset_1/centrality_threshold/`.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'src').exists() and (path / 'requirements.txt').exists():
            return path
    raise FileNotFoundError('Project root not found.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
from src.models.ml_train_and_store import load_classical_dataset, load_model, top1_bank_cohort, top1_metrics

MODEL_PATH  = PROJECT_ROOT / 'src' / 'models' / 'dataset_1' / '04_a' / 'Gradient_Boosting_(tuned).joblib'
TARGETS_DIR = PROJECT_ROOT / 'src' / 'datasets' / 'dataset_1' / 'targets'
OUT_DIR     = PROJECT_ROOT / 'src' / 'data' / 'predictions' / 'dataset_1' / 'centrality_threshold'
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = 'log_systemic_risk_label'
MODEL_NAME = 'Gradient Boosting (tuned)'
EXPERIMENT = 'dataset1_temporal_train_04a_specs'
P_LABELS   = ['p5', 'p10', 'p15', 'p20', 'p25', 'p30', 'p35', 'p40']

model = load_model(MODEL_PATH)
sources = {'dataset1_baseline': TARGETS_DIR, **{f'dataset1_{p}': TARGETS_DIR / p for p in P_LABELS}}
print('Model:', MODEL_PATH.name, '| thresholds:', len(sources))

Model: Gradient_Boosting_(tuned).joblib | thresholds: 9


## Helpers (temporal split, refit per p)

In [2]:
def load_centrality(target_dir):
    df, fcols = load_classical_dataset(PROJECT_ROOT, target_col=TARGET_COL, target_dir=target_dir)
    return df.dropna(subset=[TARGET_COL]).reset_index(drop=True), fcols

def temporal_split(df):
    k = df.copy(); k['_key'] = k['year'] * 10 + k['quarter']
    tr = k[k['_key'] <= 20214].drop(columns='_key')
    va = k[(k['_key'] > 20214) & (k['_key'] <= 20224)].drop(columns='_key')
    te = k[(k['_key'] > 20224) & (k['_key'] <= 20233)].drop(columns='_key')
    return tr.reset_index(drop=True), va.reset_index(drop=True), te.reset_index(drop=True)

def _metrics(y, yp):
    return {'mae': mean_absolute_error(y, yp), 'rmse': mean_squared_error(y, yp) ** 0.5}

def add_predictions(dataset_name, split_name, split_df, fitted, fcols):
    id_cols = [c for c in ['bank_id', 'year', 'quarter', 'period'] if c in split_df.columns]
    pf = split_df[id_cols + [TARGET_COL]].copy()
    pf['prediction'] = fitted.predict(split_df[fcols])
    pf['abs_error'] = (pf[TARGET_COL] - pf['prediction']).abs()
    pf.insert(0, 'split', split_name); pf.insert(0, 'model', MODEL_NAME)
    pf.insert(0, 'experiment', EXPERIMENT); pf.insert(0, 'dataset', dataset_name)
    return pf

def run_threshold(sources, load_fn):
    rows, preds = [], []
    for dataset_name, src in sources.items():
        df, fcols = load_fn(src)
        tr, va, te = temporal_split(df)
        fitted = clone(model).fit(tr[fcols], tr[TARGET_COL])
        cohort = top1_bank_cohort(tr, TARGET_COL)
        row = {'dataset': dataset_name, 'experiment': EXPERIMENT, 'model': MODEL_NAME}
        for sn, sdf in [('train', tr), ('validation', va), ('test', te)]:
            yp = fitted.predict(sdf[fcols]); m = _metrics(sdf[TARGET_COL], yp)
            t = top1_metrics(sdf, yp, cohort, TARGET_COL)
            row[f'{sn}_mae'] = m['mae']; row[f'{sn}_rmse'] = m['rmse']
            row[f'{sn}_top1_mae'] = t['mae']; row[f'{sn}_top1_rmse'] = t['rmse']
            preds.append(add_predictions(dataset_name, sn, sdf, fitted, fcols))
        rows.append(row)
    metrics_df = pd.DataFrame(rows)
    num = [c for c in metrics_df.columns if any(k in c for k in ('rmse', 'mae'))]
    metrics_df[num] = metrics_df[num].round(3)
    predictions = pd.concat(preds, ignore_index=True)
    metrics_df.to_csv(OUT_DIR / 'metrics.csv', index=False)
    predictions.to_csv(OUT_DIR / 'predictions.csv', index=False)
    print('saved ->', OUT_DIR)
    return metrics_df, predictions

## Run the sweep

In [3]:
metrics_df, predictions = run_threshold(sources, load_centrality)
display(metrics_df[['dataset', 'test_rmse', 'test_mae', 'test_top1_rmse', 'test_top1_mae']])

saved -> /Users/rubenmarques/Documents/Repositórios/Thesis/src/data/predictions/dataset_1/centrality_threshold


,dataset,test_rmse,test_mae,test_top1_rmse,test_top1_mae
0,dataset1_baseline,0.070,0.012,0.451,0.366
1,dataset1_p5,0.069,0.012,0.431,0.351
2,dataset1_p10,0.070,0.013,0.439,0.359
3,dataset1_p15,0.071,0.013,0.445,0.360
4,dataset1_p20,0.070,0.013,0.426,0.349
5,dataset1_p25,0.070,0.013,0.413,0.340
6,dataset1_p30,0.071,0.014,0.383,0.313
7,dataset1_p35,0.073,0.014,0.398,0.326
8,dataset1_p40,0.074,0.015,0.379,0.308
